# 11 — Movement, posture, and direction from DeepLabCut

Each playback is paired with at most one cage video (`playback_one_video.csv`). The soundtrack match has to clear 0.15, and an olive color blob has to stay on a bird in the left or right perch band through the onset clip. In the current pairing, 193 of 318 playbacks met both checks. The other rows stay in the table with blank scores.

SuperAnimal-Bird (`resnet_50` + `ssdlite`) then places keypoints inside each tight box. `left` and `right` are perch sides, not the numbered birds from the pilot, unless a clip is copied from that earlier run.

This notebook turns those coordinates into three scores. They are not the human ratings in `playback_data_combined.xlsx`. A playback with no stable video, or with keypoints that do not support a score, is left blank.

The clip starts two seconds before the playback onset and runs for ten seconds after it, at 10 frames per second. Scores use frames at `t >= 2 s`.

Image y points downward.

## Definitions

A keypoint is used only when SuperAnimal's likelihood is at least 0.35.

**Movement intensity (MI).** Body centroid = mean of the available points among beak, crown, feet, and tail tip, when at least two of them are present. The centroid is then median-smoothed over five frames so a single jumped keypoint is not counted as movement. Speed is the frame-to-frame distance of that smoothed centroid, in pixels per second. MI is the median speed. The table also keeps the mean and the peak. At least eight finite speeds are required.

**Body posture (BP).** Beak-to-tail length, in pixels, on frames where both ends exist. BP is the mean length. The table also keeps the sample standard deviation. At least eight lengths are required.

**Directionality (D).** Heading from the beak toward the crown, in degrees, via `atan2`. The reported direction is the circular mean, so a turn across ±180° does not average to zero. The table also keeps the circular standard deviation. At least eight headings are required.

Raw MI, BP, and D stay in the table. Each also has a pooled z-score (`MI_z`, `BP_z`, `D_z`): subtract the mean and divide by the population standard deviation of the finite values of that column, the same rule as the `_z` columns on the playback sheet. One scaler per metric, across every bird with a finite score. Blank scores stay blank. `D_z` is a linear z-score of the stored degree value, so two headings that sit on opposite sides of ±180° remain far apart after scaling. The mean and standard deviation are written to `playback_behavior_pose_zparams.csv`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

CHIRP = Path(r"D:\CHIRP")
DLC = CHIRP / "dlc_demo"
sys.path.insert(0, str(DLC))

from batch_paths import BEHAVIOR_CSV, COORDS_CSV, ONE_CSV, ZPARAMS_CSV
from pose_behavior import centroid_series, measures_for_tracks, speed_series, write_playback_table

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

In [ ]:
behavior = write_playback_table()
print(f"wrote {BEHAVIOR_CSV}  rows={len(behavior)}")
print(f"playbacks {behavior['FileName'].nunique()}")
print(f"rows with a video {int((behavior['video'].astype(str) != '').sum())}")
print(f"rows with a usable score {int(behavior['keypoints_usable'].sum())}")
display(pd.read_csv(ZPARAMS_CSV))
display(behavior.head(12))

In [ ]:
usable = behavior[behavior["keypoints_usable"] == 1].copy()
if usable.empty:
    print("No usable pose scores yet. The coordinate table is filled as SuperAnimal finishes each clip.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
    for ax, col, title in zip(
        axes,
        ["MI", "BP", "D"],
        ["Movement intensity (px/s)", "Body posture (px)", "Direction (deg)"],
    ):
        values = pd.to_numeric(usable[col], errors="coerce").dropna()
        ax.hist(values, bins=12, color="#e07a3d")
        ax.set_title(title)
        ax.set_xlabel(col)
    fig.tight_layout()
    plt.show()
    display(usable[["FileName", "video", "receiver_id", "MI", "MI_z", "BP", "BP_z", "D", "D_z"]].head(20))

## One bird, frame by frame

The histogram above is the clip summary. The curves below are the same definitions before they are averaged: centroid speed, beak-to-tail length, and heading. The vertical line is the playback onset at 2 seconds.

In [ ]:
if not COORDS_CSV.exists():
    print(f"No coordinates at {COORDS_CSV}")
else:
    coords = pd.read_csv(COORDS_CSV)
    usable_videos = set(usable["video"].astype(str)) if len(usable) else set()
    example = coords[coords["video"].astype(str).isin(usable_videos)]
    if example.empty:
        example = coords
    if example.empty:
        print("Coordinate file is empty.")
    else:
        video = example["video"].iloc[0]
        receiver = example["receiver_id"].iloc[0]
        track = example["track"].iloc[0]
        bird = example[
            (example["video"] == video)
            & (example["receiver_id"] == receiver)
            & (example["track"] == track)
        ].sort_values("frame")
        measured = measures_for_tracks(bird)
        post = bird[pd.to_numeric(bird["t_s"], errors="coerce") >= 2.0].reset_index(drop=True)
        cx, cy = centroid_series(post)
        speed = speed_series(cx, cy, fps=10.0)
        t = pd.to_numeric(post["t_s"], errors="coerce")
        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.plot(t, speed, color="#e07a3d")
        ax.axvline(2.0, color="0.4", ls="--", lw=1)
        ax.set_xlabel("Time in clip (s)")
        ax.set_ylabel("Speed (px/s)")
        ax.set_title(f"{video}  bird {receiver}  MI={measured['MI']}")
        fig.tight_layout()
        plt.show()
        print(measured)

## What was paired

`playback_one_video.csv` is the video chosen for each playback. `unmatched` means every candidate either lost the soundtrack cutoff or the color blob did not stay on a bird. Those rows are in the behavior table with blank MI, BP, and D.

In [ ]:
if ONE_CSV.exists():
    paired = pd.read_csv(ONE_CSV)
    print(paired["status"].value_counts().to_string())
else:
    print(f"Pairing table not written yet: {ONE_CSV}")